In [19]:
import anthropic
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
import os


## Set the proper working directory

In [20]:
os.chdir('../classifications')

In [ ]:
# os.getcwd()

'/home/daniel/code/emestrada_parser/classifications'

## Read the subject csv file with statements

`read_statements` will read the subject csv file with all statements, will separate them by topic and will create a csv file ready for AI to classify them.

In [22]:
def read_statements(file: str) -> None:
    df = pd.read_csv(file)
    subject = list(df["subject"].unique())[0]
    topics = sorted( list(df.topic.unique()) )

    # remove subject column
    df = df.drop("subject", axis=1)

    for t in topics:
        df_topic = df[df.topic == t]
        topic = t.lower().replace(" ", "_")
        df_topic.to_csv(f"./csv_files/{subject}_{topic}_statements_to_AI.csv", sep="|", index=False)
        print(f"Created csv file ({subject}_{topic}_statements.csv) to be processed by AI model.")
    
    print("\n\nAll topics where successfully preprocessed to be fed to AI classifier!")

In [23]:
read_statements("../csv_statements/quimica_statements.csv")

Created csv file (quimica_acido_base_statements.csv) to be processed by AI model.
Created csv file (quimica_cantidad_quimica_statements.csv) to be processed by AI model.
Created csv file (quimica_conf_electronica_statements.csv) to be processed by AI model.
Created csv file (quimica_enlace_quimico_statements.csv) to be processed by AI model.
Created csv file (quimica_equilibrio_quimico_statements.csv) to be processed by AI model.
Created csv file (quimica_formulacion_statements.csv) to be processed by AI model.
Created csv file (quimica_reactividad_organica_statements.csv) to be processed by AI model.
Created csv file (quimica_redox_statements.csv) to be processed by AI model.
Created csv file (quimica_solubilidad_statements.csv) to be processed by AI model.
Created csv file (quimica_termoquimica_statements.csv) to be processed by AI model.


All topics where successfully preprocessed to be fed to AI classifier!


## API consumption

In [24]:
load_dotenv('../etc/.env')
client = anthropic.Anthropic()

In [61]:
def classify_exercises(csv_file: str, prompt = "prompt.txt") -> list[str]:
    """
    enunciados: lista de dicts con keys año, convocatoria, ejercicio, texto
    retorna:    lista de dicts con keys año, convocatoria, ejercicio, tema, tipo_ejercicio
    """
    # get topic from the csv_file name
    # 'quimica_acido_base_statements_to_AI.csv'
    topic = (csv_file
             .split("/")[-1]
             .replace("_statements_to_AI.csv", "")
             .split("_")
    )
    topic = "_".join(topic[1:])
    
    instructions = Path(f"./instructions_md/{topic}.md")

    # load markdown file with instructions for the current topic
    system_prompt = instructions.read_text()

    # load the csv file with statements
    csv = Path(csv_file)

    # create content of the prompt:
    content=Path(prompt).read_text().format(topic = topic, csv = Path(csv_file).read_text())

    # create API query
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=8192,
        system=system_prompt,
        messages=[
            {"role": "user", "content": content}
        ]
    )

    # clean the API response
    result = [i for i in response.content[0].text.strip().splitlines()]

    return result

## Process all the csv files
Load the csv files for each topic and feed them to the API

In [26]:
csv_files = sorted([i.name for i in Path("./csv_files").iterdir() if
             i.name.endswith("to_AI.csv") ])
csv_files

['quimica_acido_base_statements_to_AI.csv',
 'quimica_cantidad_quimica_statements_to_AI.csv',
 'quimica_conf_electronica_statements_to_AI.csv',
 'quimica_enlace_quimico_statements_to_AI.csv',
 'quimica_equilibrio_quimico_statements_to_AI.csv',
 'quimica_formulacion_statements_to_AI.csv',
 'quimica_reactividad_organica_statements_to_AI.csv',
 'quimica_redox_statements_to_AI.csv',
 'quimica_solubilidad_statements_to_AI.csv',
 'quimica_termoquimica_statements_to_AI.csv']

In [32]:
os.getcwd()

'/home/daniel/code/emestrada_parser/classifications'

### Testing

In [ ]:
# equilibrio = classify_exercises("./csv_files/quimica_equilibrio_quimico_statements_to_AI.csv")

In [ ]:
# equilibrio[:10]

['# Clasificación de Ejercicios de Equilibrio Químico',
 '',
 'He analizado todos los ejercicios del archivo CSV y los he clasificado según los criterios indicados. Aquí está el resultado en formato CSV:',
 '',
 '```',
 'year|subject|topic|exam|exercise|exercise_type',
 '2000|química|cinetica|Junio|Ejercicio 3 Opción B|principio de LeChatelier',
 '2000|química|equilibrio_quimico|Reserva 1|Ejercicio 5 Opción A|cálculo de las constantes Kc y Kp',
 '2000|química|equilibrio_quimico|Reserva 1|Ejercicio 3 Opción B|principio de LeChatelier',
 '2000|química|cinetica|Reserva 1|Ejercicio 4 Opción B|factores que afectan a la velocidad']

In [ ]:
# equilibrio_df = equilibrio[5:]

In [ ]:
# with open("equilibrio.csv", "w") as file:
#     file.writelines("\n".join(equilibrio_df))

In [ ]:
# df_eq = pd.read_csv("equilibrio.csv", sep="|")

In [ ]:
# df_eq

,year,subject,topic,exam,exercise,exercise_type
0,2000,química,cinetica,Junio,Ejercicio 3 Opción B,principio de LeChatelier
1,2000,química,equilibrio_quimico,Reserva 1,Ejercicio 5 Opción A,cálculo de las constantes Kc y Kp
2,2000,química,equilibrio_quimico,Reserva 1,Ejercicio 3 Opción B,principio de LeChatelier
3,2000,química,cinetica,Reserva 1,Ejercicio 4 Opción B,factores que afectan a la velocidad
4,2000,química,cinetica,Reserva 2,Ejercicio 3 Opción B,verdadero o falso
...,...,...,...,...,...,...
224,2025,química,equilibrio_quimico,Julio,Ejercicio 3A,grado de disociación (pregunta)
225,2025,química,cinetica,Junio,Ejercicio 2B,NaN
226,2025,química,equilibrio_quimico,Junio,Ejercicio 3A,equilibrio heterogéneo (cálculos)
227,2025,química,equilibrio_quimico,Reserva 1,Ejercicio 2A,principio de LeChatelier


In [ ]:
# df_eq[df_eq.topic == "cinetica"]

,year,subject,topic,exam,exercise,exercise_type
0,2000,química,cinetica,Junio,Ejercicio 3 Opción B,principio de LeChatelier
3,2000,química,cinetica,Reserva 1,Ejercicio 4 Opción B,factores que afectan a la velocidad
4,2000,química,cinetica,Reserva 2,Ejercicio 3 Opción B,verdadero o falso
9,2001,química,cinetica,Junio,Ejercicio 3 Opción B,factores que afectan a la velocidad
17,2001,química,cinetica,Reserva 3,Ejercicio 3 Opción B,factores que afectan a la velocidad
28,2002,química,cinetica,Reserva 3,Ejercicio 3 Opción B,verdadero o falso
42,2004,química,cinetica,Junio,Ejercicio 3 Opción B,orden total de reacción
51,2005,química,cinetica,Junio,Ejercicio 3 Opción B,NaN
67,2006,química,cinetica,Septiembre,Ejercicio 3 Opción B,orden total de reacción
69,2007,química,cinetica,Junio,Ejercicio 3 Opción B,verdadero o falso


### Process all files

In [ ]:
# responses = {}
# for f in csv_files:
#     #get topic
#     topic = f.replace("_statements_to_AI", "").replace(".csv", "")
#     topic = topic.split("_")
    
#     subject = topic[0]
#     topic = "_".join(topic[1:])
    
#     response = classify_exercises(topic = topic,
#                                   csv_file=f"./csv_files/{f}",
#                                   instructions_md=f"./instructions_md/{topic}.md")
    
#     # add to responses dictionary
#     responses[topic] = response